# Image Captioning â€” Model Prototype
Works on **RunPod**, **Kaggle**, and **local Windows/Linux**. Run cells top-to-bottom.

## 0. Detect platform

In [ ]:
import platform, sys, os

IS_WINDOWS = platform.system() == "Windows"
IS_RUNPOD  = os.path.exists("/workspace")
IS_KAGGLE  = os.path.exists("/kaggle")

if IS_RUNPOD:
    PROJECT = "/workspace/Pattern_Recog_Project"
elif IS_KAGGLE:
    PROJECT = "/kaggle/working/Pattern_Recog_Project"
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(f"Platform : {platform.system()}")
print(f"RunPod   : {IS_RUNPOD}")
print(f"Kaggle   : {IS_KAGGLE}")
print(f"Project  : {PROJECT}")

## 1. Pull latest code & install deps

In [ ]:
import subprocess

if IS_KAGGLE:
    os.makedirs("/kaggle/working", exist_ok=True)
    if not os.path.exists(PROJECT):
        subprocess.run(["git", "clone", "https://github.com/RATSAPORN/Pattern_Recog_Project.git", PROJECT], check=True)
    else:
        subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
elif IS_RUNPOD:
    subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
else:
    print("Local machine â€” skipping git pull.")

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

In [ ]:
%pip install timm pycocoevalcap pycocotools matplotlib -q

## 2. Config â€” edit before running

In [ ]:
CHECKPOINT  = "models/best.pt"      # path to saved checkpoint
ENCODER     = "vmamba_tiny"          # must match training: vit_base | vit_small | vmamba_tiny | vmamba_small | ...
DECODER     = "transformer"          # transformer | mamba | mamba3
DECODER_DIM = 512
NUM_LAYERS  = 3
MAX_LEN     = 50

IMAGE_DIR   = "data/flickr8k/images/Flicker8k_Dataset"
ANN_JSON    = "data/flickr8k_annotations.json"
TEST_SPLIT  = "data/flickr8k/text/Flickr_8k.testImages.txt"   # 1,000 held-out test images

N_IMAGES    = 12   # number of random test images to show
COLS        = 3    # columns in the grid
SEED        = 42   # change for different random samples

## 3. Load model

In [ ]:
import torch
from src.models.predict import load_model, load_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

model, vocab = load_model(
    checkpoint_path=CHECKPOINT,
    encoder_name=ENCODER,
    decoder_name=DECODER,
    vocab_path=None,        # reads vocab_path from checkpoint automatically
    decoder_dim=DECODER_DIM,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    device=DEVICE,
)

## 4. Load ground truth captions

In [ ]:
import json
from pathlib import Path

# Ground truth captions for all images
with open(ANN_JSON) as f:
    gt_captions = json.load(f)   # {filename: [caption1, caption2, ...]}

# Load test split â€” only these 1,000 images were never seen during training
with open(TEST_SPLIT) as f:
    test_images = set(line.strip() for line in f if line.strip())

print(f"Total images with captions : {len(gt_captions)}")
print(f"Test split size            : {len(test_images)} images (never seen during training)")

## 5. Random prediction grid

In [ ]:
import random
import textwrap
import matplotlib.pyplot as plt
from PIL import Image

# Only pick from test split images (never seen during training)
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_paths = [
    p for p in Path(IMAGE_DIR).iterdir()
    if p.suffix.lower() in exts and p.name in test_images
]

random.seed(SEED)
chosen = random.sample(test_paths, min(N_IMAGES, len(test_paths)))
print(f"Showing {len(chosen)} random images from test split")

rows = (len(chosen) + COLS - 1) // COLS
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 5.5))
axes = axes.flatten() if rows * COLS > 1 else [axes]

for i, path in enumerate(chosen):
    pil_img = Image.open(path).convert("RGB")
    tensor  = load_image(str(path)).to(DEVICE)

    with torch.no_grad():
        pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

    fname = path.name
    gt = gt_captions.get(fname, ["(no ground truth)"])[0]

    axes[i].imshow(pil_img)
    axes[i].axis("off")

    pred_wrapped = "\n".join(textwrap.wrap(f"Pred: {pred}", width=40))
    gt_wrapped   = "\n".join(textwrap.wrap(f"GT:   {gt}",   width=40))

    axes[i].set_title(f"{pred_wrapped}\n{gt_wrapped}", fontsize=8, loc="left",
                      color="white",
                      bbox=dict(facecolor="black", alpha=0.6, pad=3))

for j in range(len(chosen), len(axes)):
    axes[j].axis("off")

plt.suptitle(f"{ENCODER} + {DECODER}  |  test split  |  seed={SEED}", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("predictions.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved â†’ predictions.png")

## 7. Single image â€” detailed view

## 6. Evaluate on full test split

In [ ]:
from tqdm import tqdm
from src.models.train import compute_metrics

hypotheses = {}   # {image_id: [predicted_caption]}
references  = {}  # {image_id: [gt_caption1, gt_caption2, ...]}

model.eval()
for idx, path in enumerate(tqdm(test_paths, desc="evaluating test set")):
    fname  = path.name
    tensor = load_image(str(path)).to(DEVICE)

    with torch.no_grad():
        pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

    gts = gt_captions.get(fname, [""])

    hypotheses[str(idx)] = [pred]
    references[str(idx)]  = gts          # all 5 ground truth captions

metrics = compute_metrics(hypotheses, references)

print("\nâ”€â”€ Test Set Metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
for name, score in metrics.items():
    print(f"  {name:<10} {score:.4f}")
print("â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")

## 7. Error Analysis

### 7a. Per-image BLEU-4 scores (sentence level)

In [ ]:
%pip install nltk -q
import nltk
nltk.download("punkt", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoother = SmoothingFunction().method1

# Score every test image individually
per_image = []   # list of (bleu4, path, pred, gts)

for idx, path in enumerate(tqdm(test_paths, desc="scoring")):
    pred = hypotheses[str(idx)][0]
    gts  = references[str(idx)]

    pred_tokens = pred.lower().split()
    ref_tokens  = [gt.lower().split() for gt in gts]

    bleu4 = sentence_bleu(ref_tokens, pred_tokens,
                           weights=(0.25, 0.25, 0.25, 0.25),
                           smoothing_function=smoother)
    per_image.append((bleu4, path, pred, gts))

per_image.sort(key=lambda x: x[0])   # sort ascending by BLEU-4

scores = [x[0] for x in per_image]
print(f"Per-image BLEU-4  â€”  min={min(scores):.4f}  mean={sum(scores)/len(scores):.4f}  max={max(scores):.4f}")

### 7b. Best & worst predictions

In [ ]:
def plot_grid(samples, title, color):
    n = len(samples)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for i, (bleu4, path, pred, gts) in enumerate(samples):
        pil_img = Image.open(path).convert("RGB")
        axes[i].imshow(pil_img)
        axes[i].axis("off")
        pred_w = "\n".join(textwrap.wrap(f"Pred: {pred}", width=38))
        gt_w   = "\n".join(textwrap.wrap(f"GT:   {gts[0]}", width=38))
        axes[i].set_title(f"BLEU-4={bleu4:.4f}\n{pred_w}\n{gt_w}",
                          fontsize=8, loc="left", color="white",
                          bbox=dict(facecolor=color, alpha=0.75, pad=3))
    plt.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{title.lower().replace(' ', '_')}.png", bbox_inches="tight", dpi=120)
    plt.show()

# Top 6 best
plot_grid(per_image[-6:][::-1], "Best Predictions", color="darkgreen")

# Bottom 6 worst
plot_grid(per_image[:6], "Worst Predictions", color="darkred")

### 7c. Score distribution

In [ ]:
import numpy as np

scores_arr = np.array(scores)
mean_score = scores_arr.mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scores_arr, bins=40, color="steelblue", edgecolor="white")
ax.axvline(mean_score, color="red", linestyle="--", linewidth=1.5, label=f"mean={mean_score:.4f}")
ax.set_xlabel("Sentence BLEU-4", fontsize=12)
ax.set_ylabel("Number of images", fontsize=12)
ax.set_title("Per-image BLEU-4 distribution on test split", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("score_distribution.png", bbox_inches="tight", dpi=120)
plt.show()

# Breakdown by score bucket
buckets = {"0.00â€“0.05": 0, "0.05â€“0.10": 0, "0.10â€“0.20": 0, "0.20+": 0}
for s in scores_arr:
    if s < 0.05:   buckets["0.00â€“0.05"] += 1
    elif s < 0.10: buckets["0.05â€“0.10"] += 1
    elif s < 0.20: buckets["0.10â€“0.20"] += 1
    else:          buckets["0.20+"]     += 1

print("\nBLEU-4 bucket breakdown:")
for bucket, count in buckets.items():
    bar = "â–ˆ" * (count // 10)
    print(f"  {bucket}: {count:4d} images  {bar}")

### 7d. Caption length analysis

In [ ]:
pred_lengths = [len(hypotheses[str(i)][0].split()) for i in range(len(test_paths))]
gt_lengths   = [len(references[str(i)][0].split())  for i in range(len(test_paths))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pred_lengths, bins=30, color="steelblue", edgecolor="white", alpha=0.8, label="Predicted")
axes[0].hist(gt_lengths,   bins=30, color="orange",    edgecolor="white", alpha=0.6, label="Ground truth")
axes[0].set_xlabel("Caption length (words)")
axes[0].set_ylabel("Count")
axes[0].set_title("Caption length distribution")
axes[0].legend()

# Scatter: predicted length vs gt length
axes[1].scatter(gt_lengths, pred_lengths, alpha=0.2, s=10, color="steelblue")
axes[1].plot([0, max(gt_lengths)], [0, max(gt_lengths)], "r--", linewidth=1, label="perfect match")
axes[1].set_xlabel("Ground truth length (words)")
axes[1].set_ylabel("Predicted length (words)")
axes[1].set_title("Predicted vs GT caption length")
axes[1].legend()

plt.tight_layout()
plt.savefig("length_analysis.png", bbox_inches="tight", dpi=120)
plt.show()

print(f"Predicted length  â€” mean={np.mean(pred_lengths):.1f}  std={np.std(pred_lengths):.1f}")
print(f"GT length         â€” mean={np.mean(gt_lengths):.1f}  std={np.std(gt_lengths):.1f}")

### 7e. Most common words â€” predicted vs ground truth

In [ ]:
from collections import Counter

STOPWORDS = {"a", "an", "the", "is", "in", "on", "and", "with", "of", "at", "to", "are", ""}

def top_words(texts, n=20):
    counter = Counter()
    for text in texts:
        counter.update(w for w in text.lower().split() if w not in STOPWORDS)
    return counter.most_common(n)

all_preds = [hypotheses[str(i)][0] for i in range(len(test_paths))]
all_gts   = [references[str(i)][0] for i in range(len(test_paths))]

pred_words = top_words(all_preds)
gt_words   = top_words(all_gts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

words, counts = zip(*pred_words)
axes[0].barh(words[::-1], counts[::-1], color="steelblue")
axes[0].set_title("Top 20 words â€” Predicted captions")
axes[0].set_xlabel("Frequency")

words, counts = zip(*gt_words)
axes[1].barh(words[::-1], counts[::-1], color="orange")
axes[1].set_title("Top 20 words â€” Ground truth captions")
axes[1].set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("word_frequency.png", bbox_inches="tight", dpi=120)
plt.show()

# Words the model overuses vs underuses
pred_set = {w for w, _ in pred_words}
gt_set   = {w for w, _ in gt_words}
print("In top-20 predicted but NOT in top-20 GT (overused):")
print(" ", pred_set - gt_set)
print("\nIn top-20 GT but NOT in top-20 predicted (underused):")
print(" ", gt_set - pred_set)

## 8. Inference Time Comparison across Encoders

In [ ]:
import time
import gc
import torch
import matplotlib.pyplot as plt
from src.models.predict import load_model, load_image

# ── Config ──────────────────────────────────────────────────────────────────
# Each entry: (encoder_name, checkpoint_path)
# Add/remove rows to match checkpoints you actually have saved
MODELS_TO_COMPARE = [
    ("vit_base",         "models/best.pt"),   # change path if you have separate ckpts
    ("vit_small",        "models/best.pt"),
    ("vmamba_tiny",      "models/best.pt"),
    ("vmamba_slim_tiny", "models/best.pt"),
    ("vmamba_slim",      "models/best.pt"),
    ("vmamba_small_fast","models/best.pt"),
    ("vmamba_small",     "models/best.pt"),
]

N_WARMUP  = 3    # warmup runs (GPU needs to warm up its kernels)
N_MEASURE = 20   # measured runs — average over these

# Pick a fixed single test image for fair comparison
BENCH_IMAGE = str(test_paths[0])
tensor_bench = load_image(BENCH_IMAGE).to(DEVICE)

def _peek_max_len(ckpt_path, fallback=50):
    """Read decoder.pos_embedding shape from checkpoint to get correct max_len."""
    try:
        import numpy._core.multiarray
        torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])
    except Exception:
        pass
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("model", {})
        pos_emb = state.get("decoder.pos_embedding")
        if pos_emb is not None:
            return int(pos_emb.shape[1])
    except Exception:
        pass
    return fallback

# ── Benchmark loop ───────────────────────────────────────────────────────────
results = []   # [(encoder_name, mean_ms, std_ms)]

for encoder_name, ckpt_path in MODELS_TO_COMPARE:
    print(f"Timing {encoder_name} ...", end=" ", flush=True)
    try:
        # Auto-detect max_len from checkpoint so pos_embedding shapes match
        eff_max_len = _peek_max_len(ckpt_path, fallback=MAX_LEN)

        m, v = load_model(
            checkpoint_path=ckpt_path,
            encoder_name=encoder_name,
            decoder_name=DECODER,
            vocab_path=None,
            decoder_dim=DECODER_DIM,
            num_layers=NUM_LAYERS,
            max_len=eff_max_len,
            device=DEVICE,
        )
        m.eval()

        # Warmup
        for _ in range(N_WARMUP):
            with torch.no_grad():
                m.generate(tensor_bench, v, max_len=eff_max_len, device=DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        # Measure
        times = []
        for _ in range(N_MEASURE):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                m.generate(tensor_bench, v, max_len=eff_max_len, device=DEVICE)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)   # ms

        mean_ms = sum(times) / len(times)
        std_ms  = (sum((t - mean_ms) ** 2 for t in times) / len(times)) ** 0.5
        results.append((encoder_name, mean_ms, std_ms))
        print(f"{mean_ms:.1f} ± {std_ms:.1f} ms  (max_len={eff_max_len})")

    except Exception as e:
        print(f"SKIPPED ({e})")

    finally:
        # Free GPU memory before loading next model
        try:
            del m, v
        except Exception:
            pass
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

print("Done.")

In [ ]:
if not results:
    print("No results to plot â€” check that checkpoints exist.")
else:
    names  = [r[0] for r in results]
    means  = [r[1] for r in results]
    stds   = [r[2] for r in results]

    # Sort fastest â†’ slowest
    order  = sorted(range(len(means)), key=lambda i: means[i])
    names  = [names[i] for i in order]
    means  = [means[i] for i in order]
    stds   = [stds[i]  for i in order]

    colors = ["#2ecc71" if "tiny" in n or "slim" in n else
              "#3498db" if "vit"  in n else
              "#e74c3c" for n in names]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(names, means, xerr=stds, color=colors,
                   edgecolor="white", capsize=4, height=0.55)

    # Label each bar with ms value
    for bar, mean, std in zip(bars, means, stds):
        ax.text(mean + std + 1, bar.get_y() + bar.get_height() / 2,
                f"{mean:.1f} ms", va="center", fontsize=10)

    ax.set_xlabel("Inference time per image (ms) â€” lower is better", fontsize=11)
    ax.set_title(f"Encoder inference time comparison  ({DEVICE.type.upper()}  |  {N_MEASURE} runs avg)",
                 fontsize=12)
    ax.set_xlim(0, max(means) * 1.25)

    # Legend
    from matplotlib.patches import Patch
    legend = [
        Patch(color="#2ecc71", label="VMamba small/fast variants"),
        Patch(color="#e74c3c", label="VMamba full"),
        Patch(color="#3498db", label="ViT"),
    ]
    ax.legend(handles=legend, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.savefig("inference_time_comparison.png", bbox_inches="tight", dpi=120)
    plt.show()

    # Speedup relative to slowest
    slowest = max(means)
    print(f"\nSpeedup vs slowest ({names[means.index(max(means))]}):")
    for name, mean in sorted(zip(names, means), key=lambda x: x[1]):
        print(f"  {name:<20} {mean:6.1f} ms   {slowest/mean:.1f}Ã—")

In [ ]:
# Pick any image path here
SINGLE_IMAGE = str(chosen[0])

pil_img = Image.open(SINGLE_IMAGE).convert("RGB")
tensor  = load_image(SINGLE_IMAGE).to(DEVICE)

with torch.no_grad():
    pred = model.generate(tensor, vocab, max_len=MAX_LEN, device=DEVICE)[0]

fname = Path(SINGLE_IMAGE).name
gts   = gt_captions.get(fname, ["(no ground truth)"])

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(pil_img)
ax.axis("off")

caption_text = f"Predicted:\n  {pred}\n\nGround truth (all 5):"
for k, gt in enumerate(gts[:5], 1):
    caption_text += f"\n  {k}. {gt}"

fig.text(0.5, -0.02, caption_text, ha="center", va="top",
         fontsize=10, wrap=True,
         bbox=dict(facecolor="lightyellow", alpha=0.8, pad=6))

plt.tight_layout()
plt.savefig("single_prediction.png", bbox_inches="tight", dpi=120)
plt.show()
print(f"Image   : {fname}")
print(f"Predicted: {pred}")
for k, gt in enumerate(gts[:5], 1):
    print(f"GT {k}     : {gt}")